# 01. Exploratory Data Analysis & Data Quality Audit

> **Project:** Customer Retention & Revenue-at-Risk
> **Dataset:** Online Retail II (UCI) — Dec 2009 to Dec 2011

This notebook walks through the data hygiene, cleaning transformations, and exploratory profile of the retail dataset. Every step shows the business logic behind removing duplicates, isolating guest checkouts, segregating cancellation invoices, and filtering non-merchandise fees.

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# Define repository root path
BASE_DIR = Path('..').resolve() if Path('..').resolve().name == 'customer retention revenue at risk' else Path('.').resolve()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
POWERBI_DIR = PROCESSED_DIR / 'powerbi'
REPORTS_DIR = BASE_DIR / 'reports'
print('Processed data path:', PROCESSED_DIR)

Processed data path: C:\Users\WELCOME\Desktop\customer retention revenue at risk\data\processed


## 1. Data Cleaning Audit Trail & Deduplication

The raw dataset contains 1,067,371 rows. Transactions were passed through a deterministic 5-step hygiene pipeline:
1. **Exact Deduplication:** 34,335 identical rows (same invoice, stock code, timestamp, customer) were removed.
2. **Guest Checkouts:** 234,429 rows without customer IDs were isolated.
3. **Cancellations:** 19,494 return records (`C` prefix) were segregated into `fact_returns`.
4. **Price & Quantity Integrity:** 2,130 invalid price/quantity entries were excluded.
5. **Non-Product Codes:** 2,581 administrative and service charges (postage, bank fees) were isolated into `fact_non_products`.

The final clean sales dataset retains **776,844 valid merchandise transactions** across **5,853 identified customer accounts**.

In [2]:
# Summary of Cleaning Stages
audit_summary = pd.DataFrame([
    {'Stage': '1. Raw Ingested Records', 'Row Count': 1067371, 'Delta': 0, 'Retention %': 100.0},
    {'Stage': '2. Exact Deduplication', 'Row Count': 1033036, 'Delta': -34335, 'Retention %': 96.78},
    {'Stage': '3. Registered Customer Purchases', 'Row Count': 798607, 'Delta': -234429, 'Retention %': 74.82},
    {'Stage': '4. Segregate Cancellations', 'Row Count': 779113, 'Delta': -19494, 'Retention %': 73.00},
    {'Stage': '5. Positive Price & Quantity', 'Row Count': 776983, 'Delta': -2130, 'Retention %': 72.80},
    {'Stage': '6. Filter Non-Products (Clean Sales)', 'Row Count': 776844, 'Delta': -2581, 'Retention %': 72.78},
])
display(audit_summary)

,Stage,Row Count,Delta,Retention %
0,1. Raw Ingested Records,1067371,0,100.00
1,2. Exact Deduplication,1033036,-34335,96.78
2,3. Registered Customer Purchases,798607,-234429,74.82
3,4. Segregate Cancellations,779113,-19494,73.00
4,5. Positive Price & Quantity,776983,-2130,72.80
5,6. Filter Non-Products (Clean Sales),776844,-2581,72.78


## 2. Customer Returns & Cancellations (`fact_returns`)

Invoices prefixed with `'C'` represent post-purchase refunds. Rather than netting them blindly or deleting them from the audit trail, they are isolated into `fact_returns` to track return propensities and accurately compute net customer lifetime value.

In [3]:
returns_df = pd.read_csv(PROCESSED_DIR / 'fact_returns.csv')
total_return_value = abs(returns_df['line_total'].sum())
print(f'Total cancellation rows: {len(returns_df):,}')
print(f'Total return value: £{total_return_value:,.2f}')
print(f'Unique return invoices: {returns_df["invoice"].nunique():,}')
print(f'Unique customers with returns: {returns_df["customer_id"].nunique():,}')

# Top 5 largest return line items
display(returns_df[['invoice', 'customer_id', 'stock_code', 'description', 'quantity', 'price', 'line_total']]
        .sort_values(by='line_total').head(5))

Total cancellation rows: 18,390
Total return value: £1,084,812.98
Unique return invoices: 7,901
Unique customers with returns: 2,572


,invoice,customer_id,stock_code,description,quantity,price,line_total
7616,C581484,16446,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,-168469.60
6976,C541433,12346,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,-77183.60
4532,C556445,15098,M,Manual,-1,38970.00,-38970.00
18176,C512770,17399,M,Manual,-1,25111.09,-25111.09
13274,C502262,12918,M,Manual,-1,10953.50,-10953.50


## 3. Non-Product Line Items Isolation (`fact_non_products`)

Administrative and non-merchandise line items (postage fees, manual adjustments, discount lines, test entries) represent operational overhead rather than retail items. Isolating them preserves authentic order basket sizes and merchandise revenue.

In [4]:
non_prod_df = pd.read_csv(PROCESSED_DIR / 'fact_non_products.csv')
print(f'Total non-product rows: {len(non_prod_df):,}')
print(f'Total isolated service revenue: £{non_prod_df["line_total"].sum():,.2f}')

# Group by stock code
non_prod_summary = non_prod_df.groupby('stock_code').agg(
    line_items=('line_total', 'count'),
    total_amount=('line_total', 'sum'),
    sample_desc=('description', 'first')
).sort_values(by='total_amount', ascending=False).reset_index()
display(non_prod_summary)

Total non-product rows: 2,581
Total isolated service revenue: £293,675.53


,stock_code,line_items,total_amount,sample_desc
0,M,681,151777.670,Manual
1,POST,1803,124648.040,POSTAGE
2,DOT,16,11906.360,DOTCOM POSTAGE
3,ADJUST,32,3538.520,Adjustment by john on 26/01/2010 17
4,ADJUST2,3,731.050,Adjustment by Peter on Jun 25 2010
5,BANK CHARGES,31,450.001,Bank Charges
6,D,5,397.890,Discount
7,TEST001,9,225.000,This is a test product.
8,TEST002,1,1.000,This is a test product.


## 4. Customer Dimension & Geographic Profile

Let us examine customer account distributions, country revenues, and order frequencies from `dim_customer.csv` in `powerbi/`.

In [5]:
dim_cust = pd.read_csv(POWERBI_DIR / 'dim_customer.csv')
print(f'Total customer accounts: {len(dim_cust):,}')
print(f'Gross tracked customer revenue: £{dim_cust["total_revenue"].sum():,.2f}')
print(f'Net customer revenue (after returns): £{dim_cust["net_revenue"].sum():,.2f}')

# Top 5 countries by revenue
country_summary = dim_cust.groupby('country').agg(
    customer_count=('customer_id', 'count'),
    total_gross_revenue=('total_revenue', 'sum'),
    total_net_revenue=('net_revenue', 'sum'),
    avg_recency_days=('recency_days', 'mean')
).sort_values(by='total_gross_revenue', ascending=False).head(5).reset_index()
country_summary['revenue_share_pct'] = (country_summary['total_gross_revenue'] / dim_cust['total_revenue'].sum() * 100).round(2)
display(country_summary)

Total customer accounts: 5,853


Gross tracked customer revenue: £17,081,128.72
Net customer revenue (after returns): £16,119,535.94


,country,customer_count,total_gross_revenue,total_net_revenue,avg_recency_days,revenue_share_pct
0,United Kingdom,5334,14291648.50,13487344.98,203.115486,83.67
1,EIRE,3,595601.08,552540.30,58.000000,3.49
2,Netherlands,22,549773.41,544084.40,339.636364,3.22
3,Germany,106,383191.95,370131.40,131.877358,2.24
4,France,94,310081.56,281358.86,121.808511,1.82


## 5. Summary & Transition to Analytical Modeling

- **Data Quality Validated:** Deduplication, cancellation isolation, and non-product filtering produce a robust, reconciled data foundation.
- **Star Schema Modeled:** `dim_customer`, `fact_orders`, and `dim_date` are fully populated and reconciled to £17,081,128.74 gross sales.
- **Next Step:** Open `02_analysis.ipynb` for monthly cohort retention curves, behavioral RFM segmentation, and annualized Revenue-at-Risk modeling.